In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc
from pyomo.environ import (
    ConcreteModel, Set, Var, NonNegativeReals, Binary,
    Constraint, Objective, maximize, SolverFactory, value
)


In [12]:
##reads the specified 
base_dir = Path()
input_file = base_dir/"Technical Question - Attachment 2.xlsx"
output_file = base_dir/"output.xlsx"

In [13]:
#prepare data
df_hh = pd.read_excel(input_file, sheet_name="Half-hourly data")
df_hh = df_hh.rename(columns={df_hh.columns[0]:"timestamp"})
df_hh["timestamp"] = pd.to_datetime(df_hh["timestamp"], dayfirst=True, errors = "coerce")
df_hh = df_hh.dropna(subset=["timestamp"]).copy()
df_hh = df_hh.sort_values("timestamp").reset_index(drop=True)
df_hh["date"] = df_hh["timestamp"].dt.floor("D")

#prices M1 and M2
df_hh["p1"] = df_hh["Market 1 Price [£/MWh]"].astype(float)
df_hh["p2"] = df_hh["Market 2 Price [£/MWh]"].astype(float)

#daily data: market 3
df_d = pd.read_excel(input_file, sheet_name="Daily data")
df_d = df_d.rename(columns={df_d.columns[0]:"date"})
df_d["date"]= pd.to_datetime(df_d["date"], dayfirst = True, errors = "coerce").dt.floor("D")
df_d = df_d.dropna(subset=["date"]).copy()
df_d["p3"] = df_d["Market 3 Price [£/MWh]"].astype(float)

In [14]:
def solve_year (df_hh_year, df_d_year, year):

    #Parameters to be considered
    dt = 0.5 #hour

    E_MAX = 4.0
    P_CH_MAX = 2.0
    P_DIS_MAX = 2.0
    ETA_C = 0.95
    ETA_D = 0.95

    SOC_INIT = 2.0 #0.5 * E_MAX
    FIXED_OPEX_PER_YEAR = 5000.0 

    #daily price
    p3_map = dict(zip(df_d_year["date"], df_d_year["p3"]))

    #using only daily prices
    df_hh_year = df_hh_year[df_hh_year["date"].isin(p3_map.keys())].copy()
    df_hh_year = df_hh_year.sort_values("timestamp").reset_index(drop=True)

    #indexing
    T = list(range(len(df_hh_year)))
    date_of_t = df_hh_year["date"].tolist()
    dates = sorted(df_hh_year["date"].unique())

    p1 = df_hh_year["p1"].tolist()
    p2 = df_hh_year["p2"].tolist()
    
    #model
    m = ConcreteModel()
    m.T = Set(initialize=T, ordered=True)
    m.D = Set(initialize=dates, ordered=True)

    m.ch1 = Var(m.T, within = NonNegativeReals)
    m.dist1 = Var(m.T, within = NonNegativeReals)
    m.ch2 = Var(m.T, within = NonNegativeReals)
    m.dist2 = Var(m.T, within = NonNegativeReals)

    m.ch3 = Var(m.D, within = NonNegativeReals)
    m.dist3 = Var(m.D, within = NonNegativeReals)

    m.soc = Var(m.T, within = NonNegativeReals)
    m.z = Var(m.T, within=Binary)

    #power limits + exclusivity
    def charge_limit_rule(m, t):
        d = date_of_t[t] 
        return m.ch1[t] + m.ch2[t] + m.ch3[d] <= P_CH_MAX * m.z[t]

    def discharge_limit_rule(m, t): 
        d = date_of_t[t] 
        return m.dist1[t] + m.dist2[t] + m.dist3[d] <= P_DIS_MAX * (1 - m.z[t])

    m.charge_limit = Constraint(m.T, rule=charge_limit_rule)
    m.discharge_limit = Constraint(m.T, rule=discharge_limit_rule)

    #soc bounds
    m.soc_upper = Constraint(m.T, rule=lambda m, t: m.soc[t] <= E_MAX)
    
    #Soc dynamics
    def soc_dyn_rule(m, t):
        d = date_of_t[t]
        ch_tot = m.ch1[t] +m.ch2[t] + m.ch3[d]
        dist_tot = m.dist1[t] + m.dist2[t] + m.dist3[d]

        if t == 0:
            return m.soc[t] == SOC_INIT + (ETA_C*dt)*ch_tot - (dt/ETA_D)*dist_tot
        return m.soc[t] == m.soc[t-1] + (ETA_C*dt)*ch_tot - (dt/ETA_D)*dist_tot

    m.soc_dyn = Constraint(m.T, rule=soc_dyn_rule)

    #terminal SOC
    m.soc_terminal = Constraint(expr=m.soc[max(T)] == SOC_INIT)

    #objetive
    def obj_rule(m):
        return sum(
            dt *(
                p1[t] * (m.dist1[t] - m.ch1[t]) +
                p2[t] * (m.dist2[t] - m.ch2[t]) +
                p3_map[date_of_t[t]] * (m.dist3[date_of_t[t]] - m.ch3[date_of_t[t]])
            )
            for t in T
        )
    m.obj = Objective(rule=obj_rule, sense=maximize)

    #solve
    solver = SolverFactory("highs")
    solver.options["time_limit"] = 240
    res = solver.solve(m)

    #export only required outputs
    out = pd.DataFrame({
        "timestamp": df_hh_year["timestamp"].values,
        "charge_mw": [value(m.ch1[t] + m.ch2[t] + m.ch3[date_of_t[t]]) for t in T],
        "discharge_mw": [value(m.dist1[t] + m.dist2[t] + m.dist3[date_of_t[t]]) for t in T],
        "soc_mwh": [value(m.soc[t]) for t in T],
    })

    #yearly profit
    out["profit_hh"] = dt * (
        df_hh_year["p1"].values * (out["discharge_mw"]*0 + [value(m.dist1[t] - m.ch1[t]) for t 
in T]) +
        df_hh_year["p2"].values * (out["discharge_mw"]*0 + [value(m.dist2[t] - m.ch2[t]) for t 
in T]) +
        pd.Series(df_hh_year["date"].values).map(p3_map).values * (out["discharge_mw"]*0 +
[value(m.dist3[date_of_t[t]] - m.ch3[date_of_t[t]]) for t in T])
    ) 
    profit_gross = out["profit_hh"].sum()
    profit_net = profit_gross - 5000.0

    summary = pd.DataFrame([{
        "year": year,
        "profit_gross_gpb": profit_gross,
        "profit_net_gbp": profit_net
    }])

    #free memory
    del m
    gc.collect()

    return out, summary

In [15]:
## run (2018 - 2020)
dispatch_all = []
summary_all = []

for year in [2018,2020,2019]: #2019, 2020]:
    df_hh_year = df_hh[df_hh["timestamp"].dt.year == year].copy()
    df_d_year = df_d[df_d["date"].dt.year == year].copy()

    out_year, summary_year = solve_year(df_hh_year, df_d_year, year)

    dispatch_all.append(out_year)
    summary_all.append(summary_year)

dispatch = pd.concat(dispatch_all).reset_index(drop=True)
profits = pd.concat(summary_all).reset_index(drop=True)

#save excel
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    dispatch.to_excel(writer, sheet_name="half_hourly_dispacth", index=False)
    profits.to_excel(writer, sheet_name="yearly_profit", index=False)

print("Saved:", output_file)
print(profits)

Saved: C:\Users\9011808\OneDrive - CPFL Energia S A\Área de Trabalho\documentos\output_todos.xlsx
   year  profit_gross_gpb  profit_net_gbp
0  2018      73107.811170    68107.811170
1  2020      85798.184413    80798.184413
2  2019      71680.931393    66680.931393
